In [0]:
dbutils.widgets.removeAll()

In [0]:
dbutils.widgets.text("reconciliation_table", "oh_apm_stg.tmp.ctl_reconciliation_log_Prv")
dbutils.widgets.text("error_table", "oh_apm_stg.vendor_extracts.error_load_report_log_cpc_Stg_Prv")
dbutils.widgets.text("s3_path", "s3://gia-stg-oh-ue1-data-raw/haven/inbound/VE_EDW/weekly")

In [0]:
reconciliation_table = dbutils.widgets.get("reconciliation_table")
error_table = dbutils.widgets.get("error_table")
s3_path = dbutils.widgets.get("s3_path")

In [0]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, LongType

def safe_get(task_key, key, default):
    try:
        return dbutils.jobs.taskValues.get(taskKey=task_key, key=key, debugValue=default)
    except Exception:
        return default


gz_expected_info = safe_get("Pre_validation", "gz_expected_info", [])
gz_file_paths = safe_get("Pre_validation", "gz_file_paths", {})
missing_required_gz_files = safe_get("Pre_validation", "missing_required_gz_files", [])
start_load = safe_get("Pre_validation", "start_load", None)


from datetime import datetime
if not start_load:
    start_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


end_load = datetime.now().strftime("%Y-%m-%d %H:%M:%S")


if gz_file_paths:
    first_path = list(gz_file_paths.values())[0]
    folder_name = first_path.split("/")[-2]
    date_received = folder_name.replace("dt=", "") if folder_name.startswith("dt=") else datetime.now().strftime("%Y-%m-%d")
else:
    date_received = datetime.now().strftime("%Y-%m-%d")

error_records = []
missing_files = [name for name, _ in gz_expected_info if name not in gz_file_paths]
missing_list = missing_files + list(missing_required_gz_files)

if missing_list:
    missing_msg = f"MISSING FILE(S): {', '.join(missing_list)}"

    for file_name, _ in gz_expected_info:
        error_records.append((
            file_name,
            "" if file_name in missing_list else date_received,
            start_load,
            end_load,
            0,
            missing_msg
        ))

    error_schema = StructType([
        StructField("File_Name", StringType(), True),
        StructField("Date_Received", StringType(), True),
        StructField("Start_Load_Date", StringType(), True),
        StructField("End_Load_Date", StringType(), True),
        StructField("Row_Number", LongType(), True),
        StructField("Error_Description", StringType(), True)
    ])

    error_df = spark.createDataFrame(error_records, schema=error_schema)

    if error_table:
        error_df.write.mode("append").saveAsTable(error_table)
        print(f"✅ Error report written to {error_table}")
        
        # Show what was written
        print("📊 Records written:")
        display(error_df)   # Use `.show()` if running outside Databricks
        dbutils.jobs.taskValues.set(key="staging_has_errors", value=True)
    else:
        print("⚠️ No error_table specified. Skipping write.")
else:
    print("✅ No missing files detected.")
    dbutils.jobs.taskValues.set(key="staging_has_errors", value=False)